# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [33]:
import pandas as pd
import numpy as np

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [34]:
df = pd.read_csv('data/AviationData.csv', low_memory=False, encoding='latin-1')

display(df.head())
df.info()

nan_counts = df.isna().sum()
nan_pct = (nan_counts / len(df) * 100).round(2)
nan_summary = pd.DataFrame({"NaN Count": nan_counts, "NaN %": nan_pct})
print(nan_summary[nan_summary["NaN Count"] > 0].sort_values("NaN Count", ascending=False))

display(df.describe())


,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,UNK,Cruise,Probable Cause,NaN
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,4.0,0.0,0.0,0.0,UNK,Unknown,Probable Cause,19-09-1996
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,NaN,NaN,...,Personal,NaN,3.0,NaN,NaN,NaN,IMC,Cruise,Probable Cause,26-02-2007
3,20001218X45448,Accident,LAX96LA321,1977-06-19,"EUREKA, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,IMC,Cruise,Probable Cause,12-09-2000
4,20041105X01764,Accident,CHI79FA064,1979-08-02,"Canton, OH",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,1.0,2.0,NaN,0.0,VMC,Approach,Probable Cause,16-04-1980


<class 'pandas.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  str    
 1   Investigation.Type      88889 non-null  str    
 2   Accident.Number         88889 non-null  str    
 3   Event.Date              88889 non-null  str    
 4   Location                88837 non-null  str    
 5   Country                 88663 non-null  str    
 6   Latitude                34382 non-null  str    
 7   Longitude               34373 non-null  str    
 8   Airport.Code            50132 non-null  str    
 9   Airport.Name            52704 non-null  str    
 10  Injury.Severity         87889 non-null  str    
 11  Aircraft.damage         85695 non-null  str    
 12  Aircraft.Category       32287 non-null  str    
 13  Registration.Number     87507 non-null  str    
 14  Make                    88826 non-null  str    
 

,Number.of.Engines,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,82805.000000,77488.000000,76379.000000,76956.000000,82977.000000
mean,1.146585,0.647855,0.279881,0.357061,5.325440
std,0.446510,5.485960,1.544084,2.235625,27.913634
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,1.000000
75%,1.000000,0.000000,0.000000,0.000000,2.000000
max,8.000000,349.000000,161.000000,380.000000,699.000000


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [35]:
print(df['Aircraft.Category'].value_counts(dropna=False))
print(df['Amateur.Built'].value_counts(dropna=False))
print(df['Event.Date'].head())
print("dtype:", df['Event.Date'].dtype)
df['Event.Date'] = pd.to_datetime(df['Event.Date'], errors='coerce')

df = df[df['Aircraft.Category'] == 'Airplane']
print(f"\nAfter Airplane filter: {df.shape}")

df = df[df['Amateur.Built'] == 'No']
print(f"After Amateur.Built filter: {df.shape}")

cutoff = pd.Timestamp('1983-01-01')
df = df[df['Event.Date'] >= cutoff]
print(f"After date filter (>= 1983): {df.shape}")

display(df['Event.Date'].describe())

Aircraft.Category
NaN                  56602
Airplane             27617
Helicopter            3440
Glider                 508
Balloon                231
Gyrocraft              173
Weight-Shift           161
Powered Parachute       91
Ultralight              30
Unknown                 14
WSFT                     9
Powered-Lift             5
Blimp                    4
UNK                      2
Rocket                   1
ULTR                     1
Name: count, dtype: int64
Amateur.Built
No     80312
Yes     8475
NaN      102
Name: count, dtype: int64
0    1948-10-24
1    1962-07-19
2    1974-08-30
3    1977-06-19
4    1979-08-02
Name: Event.Date, dtype: str
dtype: str

After Airplane filter: (27617, 31)
After Amateur.Built filter: (24417, 31)
After date filter (>= 1983): (21447, 31)


count                         21447
mean     2013-10-10 16:03:05.312631
min             1983-03-18 00:00:00
25%             2009-08-15 00:00:00
50%             2013-11-03 00:00:00
75%             2018-06-11 12:00:00
max             2022-12-26 00:00:00
Name: Event.Date, dtype: object

### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [36]:
injury_cols = ['Total.Fatal.Injuries', 'Total.Serious.Injuries',
               'Total.Minor.Injuries', 'Total.Uninjured']

print(df[injury_cols].isna().sum())
display(df[injury_cols].describe())
for col in injury_cols:
    df[col] = df[col].fillna(0)

df['Total.Passengers'] = (
    df['Total.Fatal.Injuries'] + df['Total.Serious.Injuries'] +
    df['Total.Minor.Injuries'] + df['Total.Uninjured']
)
df['Injury.Rate'] = np.where(
    df['Total.Passengers'] > 0,
    (df['Total.Fatal.Injuries'] + df['Total.Serious.Injuries']) / df['Total.Passengers'],
    np.nan
)

print(df['Injury.Rate'].describe())


Total.Fatal.Injuries      2750
Total.Serious.Injuries    2828
Total.Minor.Injuries      2544
Total.Uninjured            711
dtype: int64


,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,18697.000000,18619.000000,18903.000000,20736.000000
mean,0.740868,0.323326,0.230598,7.718364
std,6.770638,2.378395,1.753088,35.633119
min,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,1.000000
75%,0.000000,0.000000,0.000000,2.000000
max,295.000000,161.000000,200.000000,588.000000


count    20543.000000
mean         0.283971
std          0.431704
min          0.000000
25%          0.000000
50%          0.000000
75%          0.800000
max          1.000000
Name: Injury.Rate, dtype: float64


**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [37]:
print(df['Aircraft.damage'].value_counts(dropna=False))
df['Aircraft.damage'] = df['Aircraft.damage'].str.strip().str.title()
df['Aircraft.Destroyed'] = df['Aircraft.damage'].map({
    'Destroyed': 1,
    'Substantial': 0,
    'Minor': 0
})

print(df['Aircraft.damage'].value_counts(dropna=False))
print(df['Aircraft.Destroyed'].value_counts(dropna=False))

Aircraft.damage
Substantial    16990
Destroyed       2316
NaN             1227
Minor            817
Unknown           97
Name: count, dtype: int64
Aircraft.damage
Substantial    16990
Destroyed       2316
NaN             1227
Minor            817
Unknown           97
Name: count, dtype: int64
Aircraft.Destroyed
0.0    17807
1.0     2316
NaN     1324
Name: count, dtype: int64


### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [38]:
print(f"Make NaNs: {df['Make'].isna().sum()}")
print(f"Unique Makes (raw): {df['Make'].nunique()}")
display(df['Make'].value_counts().head(20))

df['Make'] = df['Make'].str.strip().str.upper()
df = df.dropna(subset=['Make'])
make_counts = df['Make'].value_counts()
valid_makes = make_counts[make_counts >= 50].index
df = df[df['Make'].isin(valid_makes)]

display(df['Make'].value_counts().head(20))

print(f"\nMakes remaining after threshold filter: {df['Make'].nunique()}")
print(f"DataFrame shape: {df.shape}")


Make NaNs: 3
Unique Makes (raw): 1332


Make
CESSNA                4867
PIPER                 2803
Cessna                2279
Piper                 1186
BOEING                1037
BEECH                 1018
Beech                  413
MOONEY                 238
Boeing                 227
CIRRUS DESIGN CORP     218
AIR TRACTOR INC        217
AIRBUS                 215
BELLANCA               158
AERONCA                149
MAULE                  144
Mooney                 125
EMBRAER                123
Air Tractor            117
LUSCOMBE                95
STINSON                 91
Name: count, dtype: int64

Make
CESSNA                7146
PIPER                 3989
BEECH                 1431
BOEING                1264
MOONEY                 363
AIRBUS                 243
CIRRUS DESIGN CORP     220
BELLANCA               219
AIR TRACTOR INC        219
MAULE                  215
AIR TRACTOR            206
AERONCA                200
CHAMPION               158
EMBRAER                153
GRUMMAN                147
LUSCOMBE               141
CIRRUS                 137
STINSON                129
MCDONNELL DOUGLAS      108
NORTH AMERICAN         106
Name: count, dtype: int64


Makes remaining after threshold filter: 36
DataFrame shape: (17892, 34)


### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [39]:
print(f"Model NaNs: {df['Model'].isna().sum()}")
print()
display(df['Model'].value_counts().head(20))

df = df.dropna(subset=['Model'])
df['Model'] = df['Model'].str.strip().str.upper()
same_model_diff_make = df.groupby('Model')['Make'].nunique()
shared = same_model_diff_make[same_model_diff_make > 1].sort_values(ascending=False)
print(f"\nModel names shared across multiple Makes: {len(shared)}")
print(shared.head(10))

df['Make.Model'] = df['Make'] + ' ' + df['Model']
print(f"\nUnique Models: {df['Model'].nunique()}")
print(f"Unique Make.Models: {df['Make.Model'].nunique()}")
print()
display(df['Make.Model'].value_counts().head(15))


Model NaNs: 13



Model
172          769
737          403
152          316
182          304
172S         276
PA28         273
172N         249
SR22         240
180          213
A36          181
172M         180
150          179
PA-18-150    175
PA-28-140    169
172P         143
140          117
172R         109
170B         107
PA-28-180    105
PA-28-161    102
Name: count, dtype: int64


Model names shared across multiple Makes: 118
Model
500      3
400      3
7AC      3
7GCBC    3
7EC      3
7ECA     3
7GCAA    3
8KCAB    3
8GCBC    3
S2R      3
Name: Make, dtype: int64

Unique Models: 2025
Unique Make.Models: 2153



Make.Model
CESSNA 172                 769
BOEING 737                 403
CESSNA 152                 316
CESSNA 182                 304
CESSNA 172S                276
PIPER PA28                 273
CESSNA 172N                249
CESSNA 180                 213
CESSNA 172M                180
CESSNA 150                 179
PIPER PA-18-150            175
PIPER PA-28-140            169
BEECH A36                  164
CIRRUS DESIGN CORP SR22    144
CESSNA 172P                143
Name: count, dtype: int64

### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [40]:
print(df['Engine.Type'].value_counts(dropna=False))
df['Engine.Type'] = df['Engine.Type'].str.strip().str.title()
engine_counts = df['Engine.Type'].value_counts()
rare_engines = engine_counts[engine_counts < 50].index
df['Engine.Type'] = df['Engine.Type'].replace(dict.fromkeys(rare_engines, np.nan))
print(df['Engine.Type'].value_counts(dropna=False))

print()

print(df['Weather.Condition'].value_counts(dropna=False))
df['Weather.Condition'] = df['Weather.Condition'].replace('UNK', np.nan)
df['Weather.Condition'] = df['Weather.Condition'].str.strip()
print(df['Weather.Condition'].value_counts(dropna=False))

print()

print(df['Number.of.Engines'].value_counts(dropna=False))
df['Number.of.Engines'] = df['Number.of.Engines'].replace(0, np.nan)
print(df['Number.of.Engines'].value_counts(dropna=False))

print()

print(df['Purpose.of.flight'].value_counts(dropna=False))
df['Purpose.of.flight'] = df['Purpose.of.flight'].str.strip().str.title()
purpose_counts = df['Purpose.of.flight'].value_counts()
rare_purposes = purpose_counts[purpose_counts < 50].index
df['Purpose.of.flight'] = df['Purpose.of.flight'].replace(
    dict.fromkeys(rare_purposes, 'Other')
)
print(df['Purpose.of.flight'].value_counts(dropna=False))

print()

print(df['Broad.phase.of.flight'].value_counts(dropna=False))
df['Broad.phase.of.flight'] = df['Broad.phase.of.flight'].str.strip().str.title()
df['Broad.phase.of.flight'] = df['Broad.phase.of.flight'].replace(
    {'Unknown': np.nan, 'None': np.nan}
)
print(df['Broad.phase.of.flight'].value_counts(dropna=False))


Engine.Type
Reciprocating      12835
NaN                 3214
Turbo Prop           931
Turbo Fan            701
Unknown              105
Turbo Jet             71
Geared Turbofan       12
Turbo Shaft            9
UNK                    1
Name: count, dtype: int64
Engine.Type
Reciprocating    12835
NaN               3236
Turbo Prop         931
Turbo Fan          701
Unknown            105
Turbo Jet           71
Name: count, dtype: int64

Weather.Condition
VMC    14295
NaN     2417
IMC      905
Unk      186
UNK       76
Name: count, dtype: int64
Weather.Condition
VMC    14295
NaN     2493
IMC      905
Unk      186
Name: count, dtype: int64

Number.of.Engines
1.0    13222
2.0     2470
NaN     2089
4.0       67
3.0       26
0.0        5
Name: count, dtype: int64
Number.of.Engines
1.0    13222
2.0     2470
NaN     2094
4.0       67
3.0       26
Name: count, dtype: int64

Purpose.of.flight
Personal                     9844
NaN                          3047
Instructional                2410
Ae

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [41]:
non_null_counts = df.notna().sum().sort_values()
print(non_null_counts)
print()

cols_to_drop = ["Schedule", "Air.carrier", "Latitude", "Longitude",
                "Airport.Code", "Airport.Name", "Report.Status", "FAR.Description"]
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

print(f"Columns kept: {len(df.columns)}")
print(df.columns.tolist())
print(f"Final DataFrame shape: {df.shape}")


Schedule                   2139
Broad.phase.of.flight      2441
Air.carrier                8448
Airport.Code              11648
Airport.Name              11754
Report.Status             14094
Engine.Type               14643
Purpose.of.flight         14832
Weather.Condition         15386
Number.of.Engines         15785
Longitude                 15978
Latitude                  15981
Aircraft.Destroyed        16733
Aircraft.damage           16815
Publication.Date          17091
Injury.Rate               17103
Injury.Severity           17162
FAR.Description           17535
Registration.Number       17715
Location                  17875
Country                   17878
Accident.Number           17879
Event.Date                17879
Aircraft.Category         17879
Model                     17879
Make                      17879
Total.Fatal.Injuries      17879
Amateur.Built             17879
Total.Uninjured           17879
Total.Minor.Injuries      17879
Investigation.Type        17879
Event.Id

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [ ]:
df.to_csv('data/cleaned_aviation_data.csv', index=False)